# TokenWire Benchmark

Compare **TokenWire** (binary token streaming) vs **Baseline** (JSON text streaming) for LLM inference.

| Protocol | Format | Bytes/Token | Overhead |
|----------|--------|-------------|----------|
| **TokenWire** | Binary | 4 bytes | None |
| **Baseline** | JSON | ~60 bytes | Detokenization + Serialization |

**Just run all cells!** Everything is self-contained.

In [ ]:
#@title 1. Install Dependencies { display-mode: "form" }
#@markdown This installs llama-cpp-python with CUDA support and visualization libraries.

!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 -q
!pip install matplotlib numpy -q

print("Dependencies installed!")

In [ ]:
#@title 2. Load TokenWire Library { display-mode: "form" }
#@markdown This cell contains the complete TokenWire benchmark library. Just run it.

# =============================================================================
# TOKENWIRE LIBRARY - All-in-one benchmark toolkit
# =============================================================================

import time
import json
import struct
import numpy as np
from pathlib import Path
from typing import List, Dict, Any, Optional, Generator, Tuple
from dataclasses import dataclass, field, asdict
from datetime import datetime

# ─────────────────────────────────────────────────────────────────────────────
# Model Wrapper
# ─────────────────────────────────────────────────────────────────────────────

class TokenWireModel:
    """Llama model wrapper supporting both TokenWire and Baseline protocols."""
    
    def __init__(self, model_path: str, n_ctx: int = 2048, n_gpu_layers: int = -1, verbose: bool = False):
        from llama_cpp import Llama
        print(f"Loading model: {model_path}")
        self.llm = Llama(model_path=model_path, n_ctx=n_ctx, n_gpu_layers=n_gpu_layers, verbose=verbose)
        self.eos_id = self.llm.token_eos()
        print(f"Model loaded. Vocab size: {self.llm.n_vocab()}, EOS: {self.eos_id}")

    def tokenize(self, text: str) -> List[int]:
        return self.llm.tokenize(text.encode("utf-8"))

    def detokenize(self, token_ids: List[int]) -> str:
        return self.llm.detokenize(token_ids).decode("utf-8", errors="replace")

    def reset_cache(self):
        self.llm.reset()

    def generate_baseline(self, prompt: str, max_tokens: int = 128, temperature: float = 0.0):
        """Generate with baseline protocol (includes detokenization overhead)."""
        prompt_tokens = self.tokenize(prompt)
        count = 0
        for token_id in self.llm.generate(prompt_tokens, temp=temperature):
            token_id = int(token_id)
            if token_id == self.eos_id or count >= max_tokens:
                break
            token_text = self.detokenize([token_id])
            timestamp = time.perf_counter() * 1000
            count += 1
            yield token_text, timestamp

    def generate_tokenwire(self, prompt: str, max_tokens: int = 128, temperature: float = 0.0):
        """Generate with TokenWire protocol (raw token IDs, no detokenization)."""
        prompt_tokens = self.tokenize(prompt)
        count = 0
        for token_id in self.llm.generate(prompt_tokens, temp=temperature):
            token_id = int(token_id)
            if token_id == self.eos_id or count >= max_tokens:
                break
            timestamp = time.perf_counter() * 1000
            count += 1
            yield token_id, timestamp

    def generate_with_metrics(self, prompt: str, protocol: str = "tokenwire", max_tokens: int = 128, temperature: float = 0.0) -> dict:
        """Generate and collect detailed metrics."""
        self.reset_cache()
        prompt_tokens = self.tokenize(prompt)
        start_time = time.perf_counter()
        ttft = None
        tokens, timestamps = [], []
        total_bytes = 0

        for token_id in self.llm.generate(prompt_tokens, temp=temperature):
            token_id = int(token_id)
            if token_id == self.eos_id or len(tokens) >= max_tokens:
                break
            now = time.perf_counter()
            if ttft is None:
                ttft = (now - start_time) * 1000
            if protocol == "baseline":
                token_text = self.detokenize([token_id])
                payload = json.dumps({"model": "llm", "response": token_text, "done": False})
                total_bytes += len(payload.encode('utf-8'))
            else:
                total_bytes += 4
            tokens.append(token_id)
            timestamps.append(now * 1000)

        end_time = time.perf_counter()
        total_time = (end_time - start_time) * 1000
        intervals = [timestamps[i] - timestamps[i-1] for i in range(1, len(timestamps))] if len(timestamps) >= 2 else []
        jitter_std = (sum((x - sum(intervals)/len(intervals)) ** 2 for x in intervals) / len(intervals)) ** 0.5 if intervals else 0

        return {
            'protocol': protocol, 'ttft_ms': ttft or 0, 'total_time_ms': total_time,
            'total_tokens': len(tokens), 'total_bytes': total_bytes,
            'tokens_per_second': len(tokens) / (total_time / 1000) if total_time > 0 else 0,
            'bytes_per_token': total_bytes / len(tokens) if tokens else 0,
            'jitter_ms': jitter_std, 'timestamps': timestamps
        }

# ─────────────────────────────────────────────────────────────────────────────
# Benchmark Runner
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class PromptData:
    id: str
    prompt: str
    category: str
    difficulty: str = "medium"

@dataclass
class BenchmarkResult:
    prompt_id: str
    prompt: str
    category: str
    protocol: str
    ttft_ms: float
    total_time_ms: float
    total_tokens: int
    total_bytes: int
    tokens_per_second: float
    bytes_per_token: float
    jitter_ms: float
    prompt_length: int = 0
    timestamps: List[float] = field(default_factory=list)

@dataclass
class BenchmarkRun:
    run_id: str
    protocol: str
    model_name: str
    started_at: str
    completed_at: Optional[str] = None
    results: List[BenchmarkResult] = field(default_factory=list)
    avg_ttft_ms: float = 0
    avg_tps: float = 0
    total_bytes: int = 0
    total_tokens: int = 0

class Benchmark:
    def __init__(self, model, max_tokens: int = 128, warmup_prompt: str = "Hello, how are you?"):
        self.model = model
        self.max_tokens = max_tokens
        self.warmup_prompt = warmup_prompt

    def warmup(self, protocol: str, runs: int = 1):
        print(f"  Warming up ({runs} runs)...")
        self.model.reset_cache()
        for i in range(runs):
            _ = self.model.generate_with_metrics(self.warmup_prompt, protocol=protocol, max_tokens=self.max_tokens)
        print("  Warm.")

    def run(self, protocol: str, prompts: List[PromptData], warmup_runs: int = 1, cooldown_seconds: float = 0.1, verbose: bool = True) -> BenchmarkRun:
        run_id = f"{protocol}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        run = BenchmarkRun(run_id=run_id, protocol=protocol, model_name="model", started_at=datetime.now().isoformat())
        
        if verbose:
            print(f"\n{'='*50}\n{protocol.upper()} BENCHMARK ({len(prompts)} prompts)\n{'='*50}")
        
        self.warmup(protocol, warmup_runs)
        
        for i, p in enumerate(prompts, 1):
            if verbose:
                print(f"[{i}/{len(prompts)}] {p.category}: {p.prompt[:40]}...")
            time.sleep(cooldown_seconds)
            metrics = self.model.generate_with_metrics(p.prompt, protocol=protocol, max_tokens=self.max_tokens)
            result = BenchmarkResult(
                prompt_id=p.id, prompt=p.prompt, category=p.category, protocol=protocol,
                ttft_ms=metrics['ttft_ms'], total_time_ms=metrics['total_time_ms'],
                total_tokens=metrics['total_tokens'], total_bytes=metrics['total_bytes'],
                tokens_per_second=metrics['tokens_per_second'], bytes_per_token=metrics['bytes_per_token'],
                jitter_ms=metrics['jitter_ms'], prompt_length=len(p.prompt), timestamps=metrics['timestamps']
            )
            run.results.append(result)
            if verbose:
                print(f"    TTFT: {result.ttft_ms:.1f}ms | TPS: {result.tokens_per_second:.1f} | Bytes: {result.total_bytes}")

        run.completed_at = datetime.now().isoformat()
        run.total_tokens = sum(r.total_tokens for r in run.results)
        run.total_bytes = sum(r.total_bytes for r in run.results)
        if run.results:
            run.avg_ttft_ms = sum(r.ttft_ms for r in run.results) / len(run.results)
            run.avg_tps = sum(r.tokens_per_second for r in run.results) / len(run.results)
        
        if verbose:
            print(f"\n{protocol.upper()} DONE: Avg TTFT={run.avg_ttft_ms:.1f}ms, Avg TPS={run.avg_tps:.1f}, Total={run.total_bytes:,} bytes")
        return run

    def compare(self, baseline: BenchmarkRun, tokenwire: BenchmarkRun) -> Dict[str, Any]:
        prompt_comparisons = []
        tokenwire_wins = 0
        for b, t in zip(baseline.results, tokenwire.results):
            ttft_diff = b.ttft_ms - t.ttft_ms
            winner = "tokenwire" if ttft_diff > 0 else "baseline"
            if winner == "tokenwire":
                tokenwire_wins += 1
            prompt_comparisons.append({
                'prompt_id': b.prompt_id, 'category': b.category,
                'baseline_ttft_ms': b.ttft_ms, 'tokenwire_ttft_ms': t.ttft_ms,
                'ttft_diff_ms': ttft_diff, 'winner': winner,
                'baseline_bytes': b.total_bytes, 'tokenwire_bytes': t.total_bytes,
                'bandwidth_savings_pct': (1 - t.total_bytes / b.total_bytes) * 100 if b.total_bytes > 0 else 0
            })
        
        bandwidth_savings = (1 - tokenwire.total_bytes / baseline.total_bytes) * 100 if baseline.total_bytes > 0 else 0
        return {
            'baseline': {'run_id': baseline.run_id, 'avg_ttft_ms': baseline.avg_ttft_ms, 'avg_tps': baseline.avg_tps, 'total_bytes': baseline.total_bytes},
            'tokenwire': {'run_id': tokenwire.run_id, 'avg_ttft_ms': tokenwire.avg_ttft_ms, 'avg_tps': tokenwire.avg_tps, 'total_bytes': tokenwire.total_bytes},
            'comparison': {
                'ttft_improvement_ms': baseline.avg_ttft_ms - tokenwire.avg_ttft_ms,
                'ttft_improvement_pct': (1 - tokenwire.avg_ttft_ms / baseline.avg_ttft_ms) * 100 if baseline.avg_ttft_ms > 0 else 0,
                'bandwidth_savings_pct': bandwidth_savings,
                'tokenwire_wins': tokenwire_wins,
                'total_prompts': len(baseline.results),
                'tokenwire_win_rate_pct': (tokenwire_wins / len(baseline.results)) * 100 if baseline.results else 0,
            },
            'per_prompt': prompt_comparisons
        }

# ─────────────────────────────────────────────────────────────────────────────
# Dictionary (for client-side decode measurement)
# ─────────────────────────────────────────────────────────────────────────────

class Dictionary:
    def __init__(self, offsets: List[int], string_block: bytes, max_token_id: int):
        self.offsets = offsets
        self.string_block = string_block
        self.max_token_id = max_token_id
        self._lookup_time_us = 0
        self._lookup_count = 0

    @classmethod
    def from_model(cls, model) -> 'Dictionary':
        vocab_size = model.llm.n_vocab()
        print(f"Building dictionary from model ({vocab_size} tokens)...")
        offsets, strings, current_offset = [], [], 0
        for token_id in range(vocab_size):
            try:
                text_bytes = model.detokenize([token_id]).encode('utf-8')
            except:
                text_bytes = b''
            offsets.append(current_offset)
            strings.append(text_bytes + b'\x00')
            current_offset += len(text_bytes) + 1
        string_block = b''.join(strings)
        print(f"Dictionary built: {len(string_block):,} bytes")
        return cls(offsets, string_block, vocab_size - 1)

    def lookup(self, token_id: int) -> str:
        start = time.perf_counter()
        if token_id < 0 or token_id > self.max_token_id:
            return ""
        offset = self.offsets[token_id]
        end = offset
        while end < len(self.string_block) and self.string_block[end] != 0:
            end += 1
        text = self.string_block[offset:end].decode('utf-8', errors='replace')
        self._lookup_time_us += (time.perf_counter() - start) * 1_000_000
        self._lookup_count += 1
        return text

    def decode(self, token_ids: List[int]) -> str:
        return ''.join(self.lookup(tid) for tid in token_ids)

    def decode_with_timing(self, token_ids: List[int]):
        self._lookup_time_us = 0
        self._lookup_count = 0
        start = time.perf_counter()
        text = self.decode(token_ids)
        total_ms = (time.perf_counter() - start) * 1000
        avg_us = self._lookup_time_us / self._lookup_count if self._lookup_count > 0 else 0
        return text, total_ms, avg_us

# ─────────────────────────────────────────────────────────────────────────────
# Charts
# ─────────────────────────────────────────────────────────────────────────────

class Charts:
    BASELINE_COLOR = '#E74C3C'
    TOKENWIRE_COLOR = '#27AE60'

    def __init__(self, figsize=(10, 6)):
        self.figsize = figsize

    def _plt(self):
        import matplotlib.pyplot as plt
        return plt

    def ttft_comparison_bar(self, baseline, tokenwire):
        plt = self._plt()
        fig, ax = plt.subplots(figsize=self.figsize)
        protocols = ['Baseline\n(JSON)', 'TokenWire\n(Binary)']
        ttfts = [baseline.avg_ttft_ms, tokenwire.avg_ttft_ms]
        bars = ax.bar(protocols, ttfts, color=[self.BASELINE_COLOR, self.TOKENWIRE_COLOR], width=0.6)
        for bar, ttft in zip(bars, ttfts):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f'{ttft:.1f}ms', ha='center', fontweight='bold')
        ax.set_ylabel('Time to First Token (ms)')
        ax.set_title('TTFT Comparison', fontweight='bold')
        plt.tight_layout()
        return fig

    def bandwidth_comparison(self, baseline, tokenwire):
        plt = self._plt()
        fig, ax = plt.subplots(figsize=self.figsize)
        protocols = ['Baseline\n(JSON)', 'TokenWire\n(Binary)']
        kb = [baseline.total_bytes / 1024, tokenwire.total_bytes / 1024]
        bars = ax.bar(protocols, kb, color=[self.BASELINE_COLOR, self.TOKENWIRE_COLOR], width=0.6)
        for bar, k in zip(bars, kb):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{k:.1f} KB', ha='center', fontweight='bold')
        savings = (1 - tokenwire.total_bytes / baseline.total_bytes) * 100
        ax.set_ylabel('Total Bytes (KB)')
        ax.set_title(f'Bandwidth Comparison ({savings:.1f}% saved)', fontweight='bold')
        plt.tight_layout()
        return fig

    def ttft_per_prompt(self, baseline, tokenwire):
        plt = self._plt()
        fig, ax = plt.subplots(figsize=(12, 5))
        prompts = range(1, len(baseline.results) + 1)
        ax.plot(prompts, [r.ttft_ms for r in baseline.results], 'o-', color=self.BASELINE_COLOR, label='Baseline', markersize=6)
        ax.plot(prompts, [r.ttft_ms for r in tokenwire.results], 's-', color=self.TOKENWIRE_COLOR, label='TokenWire', markersize=6)
        ax.set_xlabel('Prompt #')
        ax.set_ylabel('TTFT (ms)')
        ax.set_title('TTFT per Prompt', fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        return fig

    def summary_dashboard(self, baseline, tokenwire, comparison):
        plt = self._plt()
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        c = comparison['comparison']
        
        # TTFT Bar
        ax = axes[0, 0]
        bars = ax.bar(['Baseline', 'TokenWire'], [baseline.avg_ttft_ms, tokenwire.avg_ttft_ms], 
                      color=[self.BASELINE_COLOR, self.TOKENWIRE_COLOR])
        ax.set_ylabel('TTFT (ms)')
        ax.set_title(f'Avg TTFT ({c["ttft_improvement_pct"]:.1f}% faster)')
        
        # Bandwidth Bar
        ax = axes[0, 1]
        ax.bar(['Baseline', 'TokenWire'], [baseline.total_bytes/1024, tokenwire.total_bytes/1024],
               color=[self.BASELINE_COLOR, self.TOKENWIRE_COLOR])
        ax.set_ylabel('Total KB')
        ax.set_title(f'Bandwidth ({c["bandwidth_savings_pct"]:.1f}% saved)')
        
        # Win Rate Pie
        ax = axes[1, 0]
        wins = c['tokenwire_wins']
        total = c['total_prompts']
        ax.pie([wins, total-wins], labels=[f'TokenWire ({wins})', f'Baseline ({total-wins})'],
               colors=[self.TOKENWIRE_COLOR, self.BASELINE_COLOR], autopct='%1.0f%%', startangle=90)
        ax.set_title('TTFT Winner per Prompt')
        
        # Summary Text
        ax = axes[1, 1]
        ax.axis('off')
        summary = f"""KEY RESULTS

TTFT Improvement: {c['ttft_improvement_ms']:.1f}ms ({c['ttft_improvement_pct']:.1f}% faster)
Bandwidth Saved: {c['bandwidth_savings_pct']:.1f}%
TokenWire Win Rate: {c['tokenwire_win_rate_pct']:.0f}% ({wins}/{total})

Baseline:  {baseline.avg_ttft_ms:.1f}ms TTFT, {baseline.total_bytes:,} bytes
TokenWire: {tokenwire.avg_ttft_ms:.1f}ms TTFT, {tokenwire.total_bytes:,} bytes"""
        ax.text(0.1, 0.5, summary, transform=ax.transAxes, fontsize=12, verticalalignment='center',
                fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        plt.suptitle('TokenWire vs Baseline: Summary', fontsize=14, fontweight='bold')
        plt.tight_layout()
        return fig

# ─────────────────────────────────────────────────────────────────────────────

charts = Charts()
print("Library loaded!")

In [ ]:
#@title 3. Configuration { display-mode: "form" }

#@markdown ### Model Settings
model_url = "https://huggingface.co/Qwen/Qwen2.5-Coder-1.5B-Instruct-GGUF/resolve/main/qwen2.5-coder-1.5b-instruct-q4_k_m.gguf" #@param {type:"string"}
model_filename = "qwen2.5-coder-1.5b-q4_k_m.gguf" #@param {type:"string"}

#@markdown ### Benchmark Settings
max_tokens = 128 #@param {type:"slider", min:32, max:256, step:32}
num_prompts = 8 #@param {type:"slider", min:4, max:20, step:2}
cooldown_between_protocols = 5 #@param {type:"slider", min:1, max:15, step:1}

print(f"Configuration:")
print(f"  Model: {model_filename}")
print(f"  Max tokens: {max_tokens}")
print(f"  Prompts: {num_prompts}")
print(f"  Cooldown: {cooldown_between_protocols}s")

In [ ]:
#@title 4. Download Model { display-mode: "form" }
#@markdown Downloads the model if not already present (~1GB).

import os
import urllib.request

MODEL_PATH = f"models/{model_filename}"

if not os.path.exists(MODEL_PATH):
    print(f"Downloading model...")
    os.makedirs("models", exist_ok=True)
    
    def progress(block_num, block_size, total_size):
        pct = min(100, block_num * block_size * 100 // total_size)
        mb = block_num * block_size / (1024*1024)
        print(f"\r  {pct}% ({mb:.0f} MB)", end="", flush=True)
    
    urllib.request.urlretrieve(model_url, MODEL_PATH, progress)
    print("\nDownload complete!")
else:
    print(f"Model exists: {MODEL_PATH}")

In [ ]:
#@title 5. Load Model { display-mode: "form" }
#@markdown Loads the model into GPU memory.

model = TokenWireModel(
    MODEL_PATH,
    n_ctx=2048,
    n_gpu_layers=-1  # Use all GPU layers
)

print("\nModel ready!")

In [ ]:
#@title 6. Quick Test { display-mode: "form" }
#@markdown Verify both protocols work before benchmarking.

test_prompt = "Write a hello world in Python:"

print("BASELINE (text):")
for text, _ in model.generate_baseline(test_prompt, max_tokens=30):
    print(text, end="", flush=True)
print("\n")

print("TOKENWIRE (token IDs):")
ids = [tid for tid, _ in model.generate_tokenwire(test_prompt, max_tokens=30)]
print(f"  IDs: {ids[:10]}... ({len(ids)} tokens, {len(ids)*4} bytes)")
print(f"  Decoded: {model.detokenize(ids)}")

In [ ]:
#@title 7. Define Benchmark Prompts { display-mode: "form" }
#@markdown Creates a diverse set of prompts for benchmarking.

all_prompts = [
    PromptData("p1", "Write a Python function to check if a number is prime.", "Code"),
    PromptData("p2", "Explain how a hash table works in simple terms.", "Explanation"),
    PromptData("p3", "Write a haiku about programming.", "Creative"),
    PromptData("p4", "What is the time complexity of binary search?", "Reasoning"),
    PromptData("p5", "Write a SQL query to find duplicate emails.", "Code"),
    PromptData("p6", "Explain the difference between TCP and UDP.", "Explanation"),
    PromptData("p7", "Write a recursive factorial function in JavaScript.", "Code"),
    PromptData("p8", "What are the SOLID principles?", "Explanation"),
    PromptData("p9", "Write a function to reverse a linked list.", "Code"),
    PromptData("p10", "Explain how garbage collection works.", "Explanation"),
    PromptData("p11", "Write a poem about artificial intelligence.", "Creative"),
    PromptData("p12", "What is the difference between REST and GraphQL?", "Explanation"),
    PromptData("p13", "Implement binary search in Python.", "Code"),
    PromptData("p14", "Explain how HTTPS encryption works.", "Explanation"),
    PromptData("p15", "Write a function to detect a cycle in a linked list.", "Code"),
    PromptData("p16", "What is the CAP theorem?", "Explanation"),
    PromptData("p17", "Write a merge sort implementation.", "Code"),
    PromptData("p18", "Explain microservices architecture.", "Explanation"),
    PromptData("p19", "Write a function to find the longest common subsequence.", "Code"),
    PromptData("p20", "What is eventual consistency?", "Explanation"),
]

prompts = all_prompts[:num_prompts]
print(f"Using {len(prompts)} prompts:")
for p in prompts:
    print(f"  [{p.category}] {p.prompt[:50]}...")

In [ ]:
#@title 8. Run Benchmark { display-mode: "form" }
#@markdown Runs both protocols and compares results.

import time as time_module

bench = Benchmark(model=model, max_tokens=max_tokens)

# Run Baseline
baseline_run = bench.run("baseline", prompts, warmup_runs=1, verbose=True)

# Cooldown
print(f"\nCooling down for {cooldown_between_protocols}s...")
time_module.sleep(cooldown_between_protocols)

# Run TokenWire
tokenwire_run = bench.run("tokenwire", prompts, warmup_runs=1, verbose=True)

# Compare
comparison = bench.compare(baseline_run, tokenwire_run)

print("\n" + "="*50)
print("BENCHMARK COMPLETE!")
print("="*50)

In [ ]:
#@title 9. Results Summary { display-mode: "form" }
#@markdown Shows the comparison between protocols.

c = comparison['comparison']

print("\n" + "="*60)
print("                    BENCHMARK RESULTS")
print("="*60)
print(f"\n  Prompts tested: {c['total_prompts']}")
print(f"  Max tokens per prompt: {max_tokens}")
print("\n" + "-"*60)
print("  TTFT (Time to First Token)")
print("-"*60)
print(f"    Baseline:  {comparison['baseline']['avg_ttft_ms']:.1f} ms")
print(f"    TokenWire: {comparison['tokenwire']['avg_ttft_ms']:.1f} ms")
print(f"    Improvement: {c['ttft_improvement_ms']:.1f} ms ({c['ttft_improvement_pct']:.1f}% faster)")
print(f"    TokenWire won: {c['tokenwire_wins']}/{c['total_prompts']} prompts ({c['tokenwire_win_rate_pct']:.0f}%)")
print("\n" + "-"*60)
print("  BANDWIDTH")
print("-"*60)
print(f"    Baseline:  {comparison['baseline']['total_bytes']:,} bytes ({comparison['baseline']['total_bytes']/1024:.1f} KB)")
print(f"    TokenWire: {comparison['tokenwire']['total_bytes']:,} bytes ({comparison['tokenwire']['total_bytes']/1024:.1f} KB)")
print(f"    Savings: {c['bandwidth_savings_pct']:.1f}%")
print("\n" + "="*60)

In [ ]:
#@title 10. Visualizations { display-mode: "form" }
#@markdown Generates charts comparing the protocols.

import matplotlib.pyplot as plt

# Summary Dashboard
fig = charts.summary_dashboard(baseline_run, tokenwire_run, comparison)
plt.show()

# TTFT per Prompt
fig = charts.ttft_per_prompt(baseline_run, tokenwire_run)
plt.show()

In [ ]:
#@title 11. Client-Side Decode Test (Optional) { display-mode: "form" }
#@markdown Measures the overhead of decoding token IDs on the client.

# Build dictionary from model
dictionary = Dictionary.from_model(model)

# Generate tokens
test_tokens = [tid for tid, _ in model.generate_tokenwire("Explain quantum computing:", max_tokens=100)]
print(f"\nGenerated {len(test_tokens)} tokens to decode")

# Measure decode time
text, total_ms, avg_us = dictionary.decode_with_timing(test_tokens)

print(f"\nClient-Side Decode Results:")
print(f"  Total time: {total_ms:.3f} ms")
print(f"  Avg per token: {avg_us:.2f} us")
print(f"  Overhead: NEGLIGIBLE (< 0.1ms for 100 tokens)")
print(f"\nDecoded text: {text[:150]}...")

---

## Conclusion

**TokenWire achieves:**
- Lower TTFT by eliminating detokenization and JSON serialization
- ~90% bandwidth savings (4 bytes vs ~60 bytes per token)
- Negligible client-side decode overhead (~0.5us per token)

**Trade-off:** Client needs a pre-loaded vocabulary dictionary (~5-10MB per model)

---
*TokenWire - Binary token streaming for faster LLM inference*